In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv: group by (cmdb_id, kpi_name) with count and global P95
metric_summary = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count_of_points='count', global_P95_value=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count_of_points', ascending=False)
    .head(20)
)

# distinct cmdb_id and kpi_name (limit 50)
metric_cmdbs = pd.Series(pd.unique(metric['cmdb_id']))[:50].rename('distinct_cmdb_id').to_frame()
metric_kpis = pd.Series(pd.unique(metric['kpi_name']))[:50].rename('distinct_kpi_name').to_frame()

# 2) trace.csv: group by (cmdb_id, trace_name) with count and global P95
trace_summary = (
    trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count_of_points='count', global_P95_value=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count_of_points', ascending=False)
    .head(20)
)

trace_names = pd.Series(pd.unique(trace['trace_name']))[:50].rename('distinct_trace_name').to_frame()
trace_cmdbs = pd.Series(pd.unique(trace['cmdb_id']))[:50].rename('distinct_cmdb_id').to_frame()

# 3) log.csv: group by (cmdb_id, log_name) with count and global P95
log_summary = (
    log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count_of_points='count', global_P95_value=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count_of_points', ascending=False)
    .head(20)
)

log_names = pd.Series(pd.unique(log['log_name']))[:50].rename('distinct_log_name').to_frame()
log_cmdbs = pd.Series(pd.unique(log['cmdb_id']))[:50].rename('distinct_cmdb_id').to_frame()

# 4) error_logs.csv: total count, counts per cmdb_id (top20), earliest and latest timestamps, up to 5 sample rows
error_total = len(error_logs)
error_counts = error_logs['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count').astype({'count': int})

# earliest and latest timestamps as ISO UTC datetimes
if error_total > 0:
    error_earliest = error_logs['timestamp'].min().isoformat()
    error_latest = error_logs['timestamp'].max().isoformat()
else:
    error_earliest = None
    error_latest = None

# sample rows (timestamp as ISO UTC, cmdb_id, message) up to 5
if 'message' in error_logs.columns:
    error_samples = error_logs.loc[:, ['timestamp', 'cmdb_id', 'message']].copy()
else:
    # fallback if message column missing
    error_samples = error_logs.loc[:, ['timestamp', 'cmdb_id']].copy()
    error_samples['message'] = pd.NA

# Convert timestamp to ISO UTC format (Z)
error_samples['timestamp'] = error_samples['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
error_samples = error_samples.head(5).reset_index(drop=True)

# Final compact outputs (DataFrames / scalars). The last expression determines displayed outputs.
metric_summary, metric_cmdbs, metric_kpis, trace_summary, trace_names, trace_cmdbs, log_summary, log_names, log_cmdbs, error_total, error_counts, error_earliest, error_latest, error_samples

```
Out[1]:
```
summary = (
    "Summary:\n"
    "- Metrics: the 'carts' service stands out. Its P95 values are high across multiple KPIs (CPU ~78.7, disk I/O very large, memory ~2.22e8, socket ~13.4, workload ~8.6), and it has the most metric points. "
    "The 'carts-db' also shows elevated disk I/O and memory at P95. \n"
    "- Traces: none available. \n"
    "- Logs: error counts are effectively zero (P95=0) across services; front-end shows the highest log row volume. \n"
    "- Error logs: no records.\n\n"
    "Conclusion / likely root cause: resource pressure in the carts service (and possibly carts-db) — high CPU, memory, and disk I/O (and elevated socket usage) — is the most likely cause. \n\n"
    "Suggested next steps: check carts host/pod resource usage and process metrics, investigate disk I/O and DB queries on carts-db, review socket/connection counts and network between carts and carts-db, and consider scaling or restarting the carts instances if load/resource exhaustion is confirmed."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count_of_points  global_P95_value
0          carts         cpu               25      7.868980e+01
1          carts      diskio               25      3.453547e+04
2          carts       error               25      0.000000e+00
3          carts  latency-50               25      2.074968e-02
4          carts  latency-90               25      5.143939e-02
5          carts         mem               25      2.221154e+08
6          carts      socket               25      1.338667e+01
7          carts    workload               25      8.554350e+00
8       carts-db         cpu               25      5.236400e+00
9       carts-db      diskio               25      2.307121e+06
10      carts-db         mem               25      9.077655e+07
11      carts-db      socket               25      6.976667e+00
12     catalogue         cpu               25      1.774746e-01
13     catalogue  latency-50               25      3.094681e-03
14     catalogue  latency-90               25      4.770426e-03
15     catalogue         mem               25      6.311499e+06
16     catalogue      socket               25      6.000000e+00
17     catalogue    workload               25      4.359283e+00
18  catalogue-db         cpu               25      2.107280e-01
19  catalogue-db      diskio               25      0.000000e+00,      distinct_cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5              orders
6           orders-db
7             payment
8        queue-master
9            rabbitmq
10  rabbitmq-exporter
11         session-db
12           shipping
13               user
14            user-db,   distinct_kpi_name
0               cpu
1            diskio
2             error
3        latency-50
4        latency-90
5               mem
6            socket
7          workload, Empty DataFrame
Columns: [cmdb_id, trace_name, count_of_points, global_P95_value]
Index: [], Empty DataFrame
Columns: [distinct_trace_name]
Index: [], Empty DataFrame
Columns: [distinct_cmdb_id]
Index: [],          cmdb_id         log_name  count_of_points  global_P95_value
7      front-end    log.row_count               25            2053.6
6      front-end  log.error_count               25               0.0
5      catalogue    log.row_count               25             170.6
4      catalogue  log.error_count               25               0.0
12  queue-master  log.error_count               25               0.0
13  queue-master    log.row_count               25             592.2
17      shipping    log.row_count               25              65.8
16      shipping  log.error_count               25               0.0
8         orders  log.error_count               25               0.0
9         orders    log.row_count               25             131.6
10       payment  log.error_count               25               0.0
11       payment    log.row_count               25             105.8
18          user  log.error_count               25               0.0
19          user    log.row_count               25             617.8
2       carts-db  log.error_count               24               0.0
3       carts-db    log.row_count               24               1.0
0          carts  log.error_count               23               0.0
1          carts    log.row_count               23             100.0
15    session-db    log.row_count                5               5.0
14    session-db  log.error_count                5               0.0,   distinct_log_name
0   log.error_count
1     log.row_count,   distinct_cmdb_id
0            carts
1        catalogue
2        front-end
3           orders
4          payment
5     queue-master
6         shipping
7             user
8         carts-db
9       session-db, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], None, None, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC datetimes
if metric['timestamp'].dtype != 'datetime64[ns, UTC]':
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 for each (cmdb_id, kpi_name) using the entire metric series
global_p95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95_value'})
)

# 2) Filter metric to incident window (inclusive)
start_ts = pd.to_datetime('2024-01-19 15:16:20', utc=True)
end_ts = pd.to_datetime('2024-01-19 15:46:20', utc=True)
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# 3) For each (cmdb_id, kpi_name), compute aggregates over the incident window
# Merge global P95 into windowed data for comparisons
mw = metric_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Determine anomalous points (value >= global_P95_value)
# If global_P95_value is NaN (no global p95 because no series), treat as not anomalous
mw['is_anomaly'] = False
mask_valid_p95 = mw['global_P95_value'].notna()
mw.loc[mask_valid_p95, 'is_anomaly'] = mw.loc[mask_valid_p95, 'value'] >= mw.loc[mask_valid_p95, 'global_P95_value']

# Aggregations per (cmdb_id, kpi_name)
group = mw.groupby(['cmdb_id', 'kpi_name'], dropna=False)
agg = group.agg(
    count_of_points_in_window = ('value', 'count'),
    count_of_anomalous_points = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
).reset_index()

# Attach global_P95_value
agg = agg.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Earliest and latest anomaly timestamps per group
anomalies = mw[mw['is_anomaly']].copy()
if not anomalies.empty:
    anomalies_times = (
        anomalies
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp']
        .agg(earliest_anomaly_timestamp=('min'), latest_anomaly_timestamp=('max'))
        .reset_index()
    )
    agg = agg.merge(anomalies_times, on=['cmdb_id', 'kpi_name'], how='left')
else:
    # ensure columns exist
    agg['earliest_anomaly_timestamp'] = pd.NaT
    agg['latest_anomaly_timestamp'] = pd.NaT

# Compute max_over_threshold_ratio with care for global_P95_value == 0 or NaN
def compute_ratio(row):
    p95 = row['global_P95_value']
    maxv = row['max_value_in_window']
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        if pd.isna(maxv) or maxv == 0:
            return 0.0
        return np.inf
    # normal case
    return maxv / p95

agg['max_over_threshold_ratio'] = agg.apply(compute_ratio, axis=1)

# Format earliest/latest anomaly timestamps to ISO UTC strings (Z), keep NaT as None
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
agg['latest_anomaly_timestamp'] = agg['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# 4) Keep only rows where count_of_anomalous_points > 0, sort and limit to top 20
agg_anomalous = agg[agg['count_of_anomalous_points'] > 0].copy()
agg_anomalous = agg_anomalous.sort_values(
    ['count_of_anomalous_points', 'max_over_threshold_ratio'], ascending=[False, False]
).reset_index(drop=True)

# Select and order columns for compact table
agg_anomalous_table = agg_anomalous[
    [
        'cmdb_id', 'kpi_name',
        'count_of_points_in_window', 'count_of_anomalous_points',
        'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
        'max_value_in_window', 'global_P95_value', 'max_over_threshold_ratio'
    ]
].head(20)

# 5) For top 3 component-KPIs, compute consecutive anomaly sequences within the incident window.
top3 = agg_anomalous_table.head(3)[['cmdb_id', 'kpi_name']].values.tolist()

sequences_list = []
for cmdb_id, kpi_name in top3:
    # Fetch anomaly rows for this KPI within window, ensure timestamp order
    subset = mw[(mw['cmdb_id'] == cmdb_id) & (mw['kpi_name'] == kpi_name) & (mw['is_anomaly'])].copy()
    subset = subset.sort_values('timestamp').reset_index(drop=True)
    # Limit raw anomaly rows to at most 20 as per instruction (for grouping)
    subset_limited = subset.head(20).copy()
    if subset_limited.empty:
        continue
    # Identify consecutive runs: consider consecutive if timestamp difference == 60s
    subset_limited['prev_ts'] = subset_limited['timestamp'].shift(1)
    subset_limited['diff'] = subset_limited['timestamp'] - subset_limited['prev_ts']
    freq = pd.Timedelta(seconds=60)
    subset_limited['new_seq'] = (subset_limited['diff'] != freq) | (subset_limited['diff'].isna())
    subset_limited['seq_id'] = subset_limited['new_seq'].cumsum()
    # Aggregate per sequence
    seq_agg = subset_limited.groupby('seq_id').agg(
        sequence_start_timestamp = ('timestamp', 'first'),
        sequence_end_timestamp = ('timestamp', 'last'),
        length_in_points = ('timestamp', 'count'),
        max_value_in_sequence = ('value', 'max')
    ).reset_index(drop=True)
    # Attach ratio per sequence
    p95_value = float(global_p95[(global_p95['cmdb_id']==cmdb_id) & (global_p95['kpi_name']==kpi_name)]['global_P95_value'].iloc[0])
    def ratio_seq(maxv):
        if pd.isna(p95_value):
            return np.nan
        if p95_value == 0:
            return np.inf if maxv > 0 else 0.0
        return maxv / p95_value
    seq_agg['max_over_threshold_ratio_in_sequence'] = seq_agg['max_value_in_sequence'].apply(ratio_seq)
    # Format timestamps to ISO
    seq_agg['sequence_start_timestamp'] = seq_agg['sequence_start_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    seq_agg['sequence_end_timestamp'] = seq_agg['sequence_end_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Add identifiers
    seq_agg.insert(0, 'cmdb_id', cmdb_id)
    seq_agg.insert(1, 'kpi_name', kpi_name)
    sequences_list.append(seq_agg)

# Combine sequences into one DataFrame for display (will be small: top3 only)
if sequences_list:
    sequences_df = pd.concat(sequences_list, ignore_index=True)
else:
    sequences_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','sequence_start_timestamp','sequence_end_timestamp',
        'length_in_points','max_value_in_sequence','max_over_threshold_ratio_in_sequence'
    ])

# Final compact outputs
agg_anomalous_table, sequences_df

```
Out[2]:
```
Summary of findings (based on global P95 computed from the full metric series and the incident window 2024-01-19 15:16:20–15:46:20 UTC):

What was done
- Global P95 was computed per (cmdb_id, kpi_name) using the full metric series.
- Metrics were filtered to the incident window and any point with value >= that global P95 was marked anomalous.
- Only component–KPIs with anomalous points are reported.

Key signals (compact)
- Widespread socket activity: many services show socket values >= their global P95 for the whole window (25 points each). Examples: catalogue, catalogue-db, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, user-db. For these, max_over_threshold_ratio = 1.0 (values are at the P95 level continuously).
- True spikes above P95 (higher concern):
  - carts — diskio: 2 anomalous points, max = 101,515.20, global P95 = 34,535.47, max_over_threshold_ratio ≈ 2.94 (largest ratio).
  - orders — cpu: 2 anomalous points, max = 5.0989, global P95 ≈ 2.4718, ratio ≈ 2.06.
  - queue-master — cpu: 2 anomalous points, max = 2.4409, global P95 ≈ 1.2489, ratio ≈ 1.95.
  - session-db — diskio: 2 anomalous points, max = 23,698.27, global P95 ≈ 12,926.28, ratio ≈ 1.83.
- Several "error" KPIs across components show 25 anomalous points but their global_P95 values are 0, so max_over_threshold_ratio is reported as 0 (these are nominal zero-count series where anomaly definition is corner-case due to P95==0).

Top sequences (top 3 component-KPIs and their consecutive anomaly runs)
- catalogue / socket: single long run from 2024-01-19T15:19:00Z to 2024-01-19T15:38:00Z (length 20 points), max value 6.0 (ratio 1.0).
- catalogue-db / socket: same pattern, 2024-01-19T15:19:00Z to 2024-01-19T15:38:00Z (length 20), max 4.0 (ratio 1.0).
- orders-db / socket: same pattern, 2024-01-19T15:19:00Z to 2024-01-19T15:38:00Z (length 20), max 5.0 (ratio 1.0).

Interpretation and recommended next steps
- There is a pervasive elevated socket-level activity across many services throughout the incident window. Even though many socket series are exactly at their P95 (ratio = 1), the fact they are at that high level continuously across many services is notable and could indicate increased connection counts, saturation of file/socket descriptors, or network contention.
- The most actionable anomalies (values substantially above P95) are:
  1. carts disk I/O spike (largest ratio ≈ 2.94) — investigate processes on carts writing/reading heavy I/O and any backing storage issues.
  2. orders cpu spike (ratio ≈ 2.06) and queue-master cpu spike (ratio ≈ 1.95) — check CPU-consuming tasks, unexpected load, or background jobs.
  3. session-db disk I/O elevated (ratio ≈ 1.83) — investigate DB I/O activity and potential contention.
- Suggested immediate checks:
  - Inspect socket/connection metrics (counts, listeners, TIME_WAIT/ESTABLISHED) across services and the network layer (load balancer, service mesh, or proxy).
  - On carts and session-db hosts, check disk I/O sources (hot tables, backups, compaction, log writes).
  - On orders and queue-master hosts, check process-level CPU usage and recent deployments or background jobs.
  - Correlate with logs and traces (if available) for the high-ratio KPIs to find root cause operations.

Bottom line
- The incident window shows pervasive high socket usage across many services plus distinct resource spikes: carts disk I/O (most severe), orders and queue-master CPU, and session-db disk I/O. Start by investigating socket/connection pressure and the carts disk I/O spike as highest-priority leads.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  count_of_points_in_window  count_of_anomalous_points earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  global_P95_value  max_over_threshold_ratio
0           catalogue   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             6.000000          6.000000                  1.000000
1        catalogue-db   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             4.000000          4.000000                  1.000000
2           orders-db   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             5.000000          5.000000                  1.000000
3             payment   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             5.000000          5.000000                  1.000000
4        queue-master   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             3.000000          3.000000                  1.000000
5            rabbitmq   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z            11.000000         11.000000                  1.000000
6   rabbitmq-exporter   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             1.000000          1.000000                  1.000000
7          session-db   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             3.000000          3.000000                  1.000000
8             user-db   socket                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             8.000000          8.000000                  1.000000
9               carts    error                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             0.000000          0.000000                  0.000000
10       catalogue-db   diskio                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             0.000000          0.000000                  0.000000
11          front-end    error                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             0.000000          0.000000                  0.000000
12             orders    error                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             0.000000          0.000000                  0.000000
13           rabbitmq   diskio                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             0.000000          0.000000                  0.000000
14               user    error                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z             0.000000          0.000000                  0.000000
15               user   socket                         25                         10       2024-01-19T15:19:00Z     2024-01-19T15:28:00Z            18.000000         18.000000                  1.000000
16              carts   diskio                         25                          2       2024-01-19T15:19:00Z     2024-01-19T15:20:00Z        101515.202078      34535.466762                  2.939448
17             orders      cpu                         25                          2       2024-01-19T15:20:00Z     2024-01-19T15:41:00Z             5.098910          2.471814                  2.062821
18       queue-master      cpu                         25                          2       2024-01-19T15:41:00Z     2024-01-19T15:43:00Z             2.440889          1.248942                  1.954366
19         session-db   diskio                         25                          2       2024-01-19T15:37:00Z     2024-01-19T15:43:00Z         23698.269891      12926.284318                  1.833340,         cmdb_id kpi_name sequence_start_timestamp sequence_end_timestamp  length_in_points  max_value_in_sequence  max_over_threshold_ratio_in_sequence
0     catalogue   socket     2024-01-19T15:19:00Z   2024-01-19T15:38:00Z                20                    6.0                                   1.0
1  catalogue-db   socket     2024-01-19T15:19:00Z   2024-01-19T15:38:00Z                20                    4.0                                   1.0
2     orders-db   socket     2024-01-19T15:19:00Z   2024-01-19T15:38:00Z                20                    5.0                                   1.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing variables if present; otherwise load CSVs
try:
    log
except NameError:
    log = pd.read_csv('log.csv')
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')
    error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Ensure timestamps are parsed as UTC datetimes
if log['timestamp'].dtype != 'datetime64[ns, UTC]':
    log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
if error_logs['timestamp'].dtype != 'datetime64[ns, UTC]':
    error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime('2024-01-19 15:16:20', utc=True)
end_ts = pd.to_datetime('2024-01-19 15:46:20', utc=True)

# 1) log.csv global P95 per (cmdb_id, log_name)
global_p95_log = (
    log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95_value'})
)

# Filter log rows to incident window
log_window = log[(log['timestamp'] >= start_ts) & (log['timestamp'] <= end_ts)].copy()

# Merge global P95 into windowed logs
lw = log_window.merge(global_p95_log, on=['cmdb_id', 'log_name'], how='left')

# Mark anomalies: value >= global_P95_value (only when global_P95_value is not NaN)
lw['is_anomaly'] = False
mask_valid = lw['global_P95_value'].notna()
lw.loc[mask_valid, 'is_anomaly'] = lw.loc[mask_valid, 'value'] >= lw.loc[mask_valid, 'global_P95_value']

# Aggregations per (cmdb_id, log_name)
group = lw.groupby(['cmdb_id', 'log_name'], dropna=False)
agg_log = group.agg(
    count_of_points_in_window=('value', 'count'),
    count_of_anomalous_points=('is_anomaly', 'sum'),
    max_value_in_window=('value', 'max')
).reset_index()

# Attach global P95
agg_log = agg_log.merge(global_p95_log, on=['cmdb_id', 'log_name'], how='left')

# Earliest and latest anomaly timestamps per group
anoms = lw[lw['is_anomaly']].copy()
if not anoms.empty:
    anom_times = (
        anoms
        .groupby(['cmdb_id', 'log_name'], dropna=False)['timestamp']
        .agg(earliest_anomaly_timestamp=('min'), latest_anomaly_timestamp=('max'))
        .reset_index()
    )
    agg_log = agg_log.merge(anom_times, on=['cmdb_id', 'log_name'], how='left')
else:
    agg_log['earliest_anomaly_timestamp'] = pd.NaT
    agg_log['latest_anomaly_timestamp'] = pd.NaT

# Compute max_over_threshold_ratio with care for p95==0 or NaN
def ratio_row(r):
    p95 = r['global_P95_value']
    maxv = r['max_value_in_window']
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        if pd.isna(maxv) or maxv == 0:
            return 0.0
        return np.inf
    return maxv / p95

agg_log['max_over_threshold_ratio'] = agg_log.apply(ratio_row, axis=1)

# Format timestamps to ISO UTC strings (Z), NaT -> None
agg_log['earliest_anomaly_timestamp'] = agg_log['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
agg_log['latest_anomaly_timestamp'] = agg_log['latest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and sort compact table: only keep groups with at least one anomaly (count_of_anomalous_points>0)
agg_log_anomalous = agg_log[agg_log['count_of_anomalous_points'] > 0].copy()
agg_log_anomalous = agg_log_anomalous.sort_values(
    ['count_of_anomalous_points', 'max_over_threshold_ratio'], ascending=[False, False]
).reset_index(drop=True)

log_agg_table = agg_log_anomalous[
    [
        'cmdb_id','log_name',
        'count_of_points_in_window','count_of_anomalous_points',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp',
        'max_value_in_window','global_P95_value','max_over_threshold_ratio'
    ]
].head(20)

# 2) Up to 5 sample log rows within the window for specified components
components = ['carts','carts-db','orders','queue-master','session-db','front-end']
sample_rows_list = []
for comp in components:
    comp_rows = log_window[log_window['cmdb_id'] == comp].copy()
    if comp_rows.empty:
        continue
    # Limit to 5 rows per component
    comp_rows = comp_rows.sort_values('timestamp').head(5).copy()
    comp_rows['timestamp'] = comp_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # Select relevant columns
    comp_rows = comp_rows[['timestamp','cmdb_id','log_name','value']]
    sample_rows_list.append(comp_rows)

if sample_rows_list:
    log_samples_df = pd.concat(sample_rows_list, ignore_index=True)
else:
    log_samples_df = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])

# 3) error_logs.csv: filter to incident window and produce summaries
error_window = error_logs[(error_logs['timestamp'] >= start_ts) & (error_logs['timestamp'] <= end_ts)].copy()

error_total_in_window = len(error_window)

if error_total_in_window > 0:
    error_counts_df = error_window['cmdb_id'].value_counts().head(10).rename_axis('cmdb_id').reset_index(name='count')
    error_earliest_in_window = error_window['timestamp'].min().strftime('%Y-%m-%dT%H:%M:%SZ')
    error_latest_in_window = error_window['timestamp'].max().strftime('%Y-%m-%dT%H:%M:%SZ')
    # sample rows up to 5 with timestamp ISO
    error_samples_df = error_window.sort_values('timestamp').head(5).copy()
    # Ensure message column exists
    if 'message' not in error_samples_df.columns:
        error_samples_df['message'] = pd.NA
    error_samples_df['timestamp'] = error_samples_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_samples_df = error_samples_df[['timestamp','cmdb_id','message']].reset_index(drop=True)
else:
    error_counts_df = pd.DataFrame(columns=['cmdb_id','count'])
    error_earliest_in_window = None
    error_latest_in_window = None
    error_samples_df = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Final compact outputs (display variables)
log_agg_table, log_samples_df, error_total_in_window, error_counts_df, error_earliest_in_window, error_latest_in_window, error_samples_df

```
Out[3]:
```
Summary of log & error-log analysis for the incident window 2024-01-19 15:16:20–15:46:20 UTC

What was done
- Computed global P95 per (cmdb_id, log_name) from the full log series, then flagged window rows with value >= that P95 as anomalies.
- Examined samples of log rows for carts, carts-db, orders, queue-master, session-db, front-end.
- Filtered error_logs to the same window.

Key findings
- No error log records in the incident window (error_logs count in window = 0).
- Many log.error_count series appear “anomalous” because their global P95 is 0 and the logged values are 0 — this is a corner case: value >= 0 marks every point as anomalous even though no errors occurred. Treat these as not actionable (they indicate no errors rather than true spikes).
- Real, actionable log signals (log.row_count) in the window:
  - carts / log.row_count: 23 points in window, 4 anomalous points; max 125 vs global P95 100 → ratio ≈ 1.25. (Notable increase in row count for carts.)
  - carts-db / log.row_count: 24 points, 24 anomalous; max 2 vs global P95 1 → ratio = 2.0. (Relative increase for carts-db row count.)
  - session-db / log.row_count: 5 points, 5 anomalous; max 5 = global P95 5 → ratio = 1.0 (at P95).
  - Several services (queue-master, orders, shipping, user, front-end, catalogue, payment) show small transient increases in log.row_count (max slightly above P95; ratios ~1.03–1.05), each with 1–2 anomalous points.
- Sampled log rows (up to 5 per requested component) are available and show the per-minute row_count/error_count values (no error rows present).

Interpretation
- The main log-based signals are elevated log.row_count for carts (moderate spike) and carts-db (relative increase), plus small brief increases across several services. These indicate increased request or processing volume rather than application errors.
- The many log.error_count “anomalies” are artifacts caused by global P95 == 0; they do not indicate real errors in the window.

Recommended next steps
1. Investigate the carts service and carts-db:
   - Correlate the carts log.row_count increase with the earlier-observed carts disk I/O spike (from metrics) — this combination suggests load-driven I/O activity tied to carts.
   - Check recent deployments, traffic surges, or background jobs affecting carts.
2. Inspect front-end and queue-master (they show higher row counts too) to see if increased incoming traffic drove the load across services.
3. Look at application traces (if available) and per-process metrics on carts and carts-db to find the source of higher request/IO.
4. Ignore log.error_count “anomalies” that are zero-valued; instead search logs for any non-zero error messages or stack traces (error_logs had none in-window).

Bottom line
- No error log entries during the window. Logs point to increased request/row counts (not errors), notably for carts and carts-db, which aligns with metric spikes previously observed. Investigate load/IO on carts and carts-db as highest-priority leads.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  count_of_points_in_window  count_of_anomalous_points earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  global_P95_value  max_over_threshold_ratio
0      catalogue  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
1      front-end  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
2         orders  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
3        payment  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
4   queue-master  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
5       shipping  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
6           user  log.error_count                         25                         25       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
7       carts-db    log.row_count                         24                         24       2024-01-19T15:20:00Z     2024-01-19T15:43:00Z                    2               1.0                  2.000000
8       carts-db  log.error_count                         24                         24       2024-01-19T15:20:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
9          carts  log.error_count                         23                         23       2024-01-19T15:19:00Z     2024-01-19T15:43:00Z                    0               0.0                  0.000000
10    session-db    log.row_count                          5                          5       2024-01-19T15:22:00Z     2024-01-19T15:42:00Z                    5               5.0                  1.000000
11    session-db  log.error_count                          5                          5       2024-01-19T15:22:00Z     2024-01-19T15:42:00Z                    0               0.0                  0.000000
12         carts    log.row_count                         23                          4       2024-01-19T15:26:00Z     2024-01-19T15:35:00Z                  125             100.0                  1.250000
13  queue-master    log.row_count                         25                          2       2024-01-19T15:33:00Z     2024-01-19T15:34:00Z                  621             592.2                  1.048632
14        orders    log.row_count                         25                          2       2024-01-19T15:33:00Z     2024-01-19T15:34:00Z                  138             131.6                  1.048632
15      shipping    log.row_count                         25                          2       2024-01-19T15:33:00Z     2024-01-19T15:34:00Z                   69              65.8                  1.048632
16          user    log.row_count                         25                          2       2024-01-19T15:33:00Z     2024-01-19T15:34:00Z                  647             617.8                  1.047264
17     front-end    log.row_count                         25                          2       2024-01-19T15:25:00Z     2024-01-19T15:33:00Z                 2119            2053.6                  1.031847
18     catalogue    log.row_count                         25                          2       2024-01-19T15:25:00Z     2024-01-19T15:33:00Z                  176             170.6                  1.031653
19       payment    log.row_count                         25                          2       2024-01-19T15:33:00Z     2024-01-19T15:34:00Z                  109             105.8                  1.030246,                timestamp    cmdb_id         log_name  value
0   2024-01-19T15:19:00Z      carts  log.error_count      0
1   2024-01-19T15:19:00Z      carts    log.row_count     75
2   2024-01-19T15:20:00Z      carts  log.error_count      0
3   2024-01-19T15:20:00Z      carts    log.row_count     25
4   2024-01-19T15:21:00Z      carts  log.error_count      0
..                   ...        ...              ...    ...
25  2024-01-19T15:19:00Z  front-end  log.error_count      0
26  2024-01-19T15:19:00Z  front-end    log.row_count   1361
27  2024-01-19T15:20:00Z  front-end  log.error_count      0
28  2024-01-19T15:20:00Z  front-end    log.row_count   1985
29  2024-01-19T15:21:00Z  front-end  log.error_count      0

[30 rows x 4 columns], 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], None, None, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```